# Chapter 22 — Build, overload, and explain a timing pipeline

**Goal:** understand timing and queue contracts before moving to hardware. All `*_ms` values in the core are assigned **virtual times**, not measured execution. There is no trained EEG model, human data, actuator, cloud service, or neural-task accuracy claim.

Read `TIMING_CONTRACT.md` and `ASSIGNMENT.md`. The core uses Python 3.11+ and the standard library. Matplotlib is optional for plots. This notebook contains no saved execution results.

The sequence is: define the contract → inspect nominal work → overload two bounded queues → account for missing and rejected windows → audit and save → plan a measured extension.

## 1. Load the pinned baseline

This cell downloads one immutable baseline helper when needed, verifies its bytes, and imports it from a dedicated folder. A local file with different bytes raises an error instead of being silently overwritten. Keep extensions elsewhere and restart the kernel if another `edge_timing` was previously imported. The SHA-256 check supports source identity, not device authentication or clinical validation.

In [ ]:
from pathlib import Path
from hashlib import sha256
from urllib.request import urlopen
import sys

assert sys.version_info >= (3, 11)
BASELINE_COMMIT = "8392f93b0186415a11cf465b69807bb6a3551694"
EXPECTED_SHA256 = "bf06749a4749b94589b71ed665625e2a1d376be85b718bcf8757fb1ed3819e86"
baseline_dir = Path("ch22_baseline")
baseline_dir.mkdir(exist_ok=True)
target = baseline_dir / "edge_timing.py"
url = ("https://raw.githubusercontent.com/ZagElSayed/"
       "Introduction-to-Neuroinformatics/" + BASELINE_COMMIT
       + "/chapters/ch22-edge-ai/edge_timing.py")
if target.exists():
    payload = target.read_bytes()
else:
    with urlopen(url, timeout=30) as response:
        payload = response.read()
assert sha256(payload).hexdigest() == EXPECTED_SHA256, "Baseline bytes differ; preserve edits separately."
if not target.exists():
    target.write_bytes(payload)
sys.path.insert(0, str(baseline_dir.resolve()))
import edge_timing as lab
assert Path(lab.__file__).resolve() == target.resolve(), "Restart kernel to load this baseline."
print(lab.VERSION, BASELINE_COMMIT, EXPECTED_SHA256)

## 2. Predict the timing before running

There are 4 channels at 100 Hz, 40 samples of history, and a 2-sample hop. Windows have support `(t-400,t]` ms; the oldest/newest sample timestamps span 390 ms. The newest-sample deadline is 100 ms. Assigned normal service is `6+8+2+4=20` ms and transport is 5 ms. Capacity excludes the running job.

Write your prediction: why does a 400-ms evidence interval not automatically violate this newest-sample deadline? What changes if the task requires 400 ms of evidence after an event?

In [ ]:
nominal = lab.simulate(lab.Config(policy="fifo", capacity=8, scenario="nominal"))
print(nominal["clock"])
print(nominal["summary"])
print(nominal["records"][0])

Annotate `source_ms`, arrival, start, each processing boundary, expiry, and finish in the first record. The arithmetic is a model assumption. It does not measure how long the host took to calculate mean squares or execute this cell.

In [ ]:
r = nominal["records"][0]
print({"virtual_wait_ms": r["start_ms"] - r["arrival_ms"],
       "virtual_service_ms": r["finish_ms"] - r["start_ms"],
       "virtual_newest_age_ms": r["finish_ms"] - r["source_ms"],
       "analysis_interval_ms": r["source_ms"] - r["window_start_ms"]})

## 3. Compare two bounded configurations

In `overload`, IDs 20–29 receive 80-ms inference cost. The inference budget is 40 ms; the non-cancellable job still occupies the worker for its whole modeled duration. The result is rejected at completion.

Before running, predict which configuration may keep more completed jobs and which may retain fresher waiting evidence. FIFO capacity 8 versus latest capacity 1 changes **both policy and capacity**. Do not attribute the entire difference to replacement alone.

In [ ]:
fifo = lab.simulate(lab.Config(policy="fifo", capacity=8, scenario="overload"))
latest = lab.simulate(lab.Config(policy="latest", capacity=1, scenario="overload"))
for label, bundle in [("fifo-8", fifo), ("latest-1", latest)]:
    print(label, bundle["summary"])

Now hold capacity constant. Describe which comparison answers which question, even if the results look similar under this particular schedule.

In [ ]:
fifo_1 = lab.simulate(lab.Config(policy="fifo", capacity=1, scenario="overload"))
for label, bundle in [("fifo-1", fifo_1), ("latest-1", latest)]:
    print(label, bundle["summary"])

## 4. Inspect every kind of failure

`faults` adds one missing window (ID 8), a source+130-ms arrival (ID 12), and a quality-false window (ID 15). These are window/status injections, not a wireless or EEG artifact-detector validation. Completion precedes arrival on equal virtual timestamps; pending work can dispatch before that arrival is processed.

In [ ]:
faults = lab.simulate(lab.Config(policy="latest", capacity=1, scenario="faults"))
print(faults["summary"])
for window_id in [8, 12, 15, 20, 21, 29, 30]:
    row = faults["records"][window_id]
    print({k: row[k] for k in ["id", "source_ms", "arrival_ms", "start_ms",
                              "finish_ms", "deadline_ms", "status", "command"]})

Explain why a timeout can also finish late, even though it has only one terminal reason. Why is `command=None` not a definition of a clinical fallback? What cannot a waiting-queue replacement fix about a running job?

In [ ]:
counts = faults["summary"]["status_counts"]
print("terminal total", sum(counts.values()), "expected", faults["summary"]["expected_windows"])
print("on-time release fraction", faults["summary"]["on_time_release_fraction"])
print("completed late", faults["summary"]["completed_late"])
print("no on-time release fraction", faults["summary"]["no_on_time_release_fraction"])

## 5. Plot ages together with coverage

The plot below includes only completed jobs, including timeout and late completions. Missing, replaced, and dropped windows need the status counts and all-window coverage alongside it. If Matplotlib is unavailable, install the optional requirements or complete the trace interpretation without plots.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for label, bundle in [("FIFO, capacity 8", fifo), ("Latest, capacity 1", latest)]:
    rows = [r for r in bundle["records"] if r["finish_ms"] is not None]
    axes[0].plot([r["id"] for r in rows],
                 [r["finish_ms"] - r["source_ms"] for r in rows], marker=".", label=label)
axes[0].axhline(100, color="black", linestyle="--", label="virtual deadline")
axes[0].set(xlabel="Window ID", ylabel="Completed-job age (virtual ms)")
axes[0].legend()
axes[1].bar(["FIFO-8", "Latest-1"],
            [fifo["summary"]["on_time_release_fraction"], latest["summary"]["on_time_release_fraction"]])
axes[1].set(ylim=(0, 1), ylabel="On-time releases / all expected windows")
plt.show()

Plot the waiting count to investigate **why** the ages differ. Event time is virtual; the logging function records queue transitions, and multiple events may share a timestamp.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for label, bundle in [("FIFO-8", fifo), ("Latest-1", latest)]:
    ax.step([e["time_ms"] for e in bundle["events"]],
            [e["waiting"] for e in bundle["events"]], where="post", label=label)
ax.set(xlabel="Virtual time (ms)", ylabel="Waiting windows; excludes running job")
ax.legend()
plt.show()

## 6. Run consistency checks and save a bundle

The audit checks recorded queue and release invariants. It is not independent validation of a scheduler, device, biological outcome, or synchronization mechanism. Preserve failures and investigate them rather than deleting rows.

In [ ]:
for bundle in [nominal, fifo, latest, fifo_1, faults]:
    print(lab.audit(bundle))
receipt = lab.save_bundle(faults)
print(receipt)

Read the saved bytes and check their receipt. This detects changed bytes relative to your expected digest; it does not authenticate an operator or make storage immutable.

In [ ]:
saved = Path(receipt["path"])
assert sha256(saved.read_bytes()).hexdigest() == receipt["sha256"]
print(saved.name, "matches its content receipt")

## 7. Plan a measured extension

Choose one: matched-capacity sensitivity, a measured Chapter 20 decoder path, a compatible quantized artifact, an instrumented output, a fallback-model transition, or a sustained energy experiment. Preserve the baseline and give changed code a new identity.

For host elapsed measurements, `time.perf_counter_ns()` supplies an integer-valued counter. Define the measured event, account for asynchronous accelerator completion, and keep source clock mapping separate. A nanosecond unit is not a nanosecond accuracy guarantee. Changing the assigned 8-ms cost to 4 ms is a virtual sensitivity experiment, not measured compression or acceleration.

Finish the assignment with the timing contract, annotated traces, all-window counts, conditional percentiles, saved identities, limitations, and the next evidence you would collect. There is no preferred winning policy or supplied result.

### Optional host measurement adapter

The next cell only defines a helper; it does not run your workload. After implementing a **synchronous** callable, invoke it deliberately and label the result as host-call duration. The first call is reported separately; repeated calls do not by themselves reproduce cold start. Include preprocessing and output only if they are inside your callable. Asynchronous device launches need explicit completion synchronization inside that callable. This adapter measures neither source age nor physical output.

In [ ]:
from time import perf_counter_ns

def measure_host_callable(call_once, repetitions=100):
    if type(repetitions) is not int or repetitions < 1:
        raise ValueError("repetitions must be a positive integer")
    t0 = perf_counter_ns()
    call_once()  # must include device completion if the backend is asynchronous
    first_ns = perf_counter_ns() - t0
    samples_ns = []
    for _ in range(repetitions):
        t0 = perf_counter_ns()
        call_once()
        samples_ns.append(perf_counter_ns() - t0)
    return {"boundary": "host callable start-to-return only",
            "clock": "perf_counter_ns", "first_call_ns": first_ns,
            "repeated_samples_ns": samples_ns,
            "p50_ns": lab.percentile(samples_ns, .50),
            "p95_ns": lab.percentile(samples_ns, .95),
            "p99_ns": lab.percentile(samples_ns, .99)}

# Example invocation after you implement the workload and its evaluation:
# measured = measure_host_callable(my_synchronous_decoder_call)
# This cell intentionally does not invoke the measurement helper.